# 06b — FPN: 3D Volumetric Evaluation on Test Set

**Model:** FPN / ResNet34  
**Task:** Run full 3D clinical evaluation (Dice + HD95 per class, with post-processing) on the TEST set.

This notebook invokes `src/evaluate_3d_test.py` for the FPN model and displays the results.

---

## 1 — Run 3D evaluation

In [ ]:
import subprocess, sys, os
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
python_exe = sys.executable

print(f"Python: {python_exe}")
print(f"Project: {PROJECT_ROOT}")
print()

result = subprocess.run(
    [python_exe, "-m", "src.evaluate_3d_test", "--model", "fpn"],
    cwd=str(PROJECT_ROOT),
    capture_output=False,
    text=True,
)

print(f"\nExit code: {result.returncode}")

## 2 — Load and display results

In [ ]:
import json
import pandas as pd

results_path = PROJECT_ROOT / "evaluation_outputs" / "test_3d_metrics_fpn.json"

with open(results_path) as f:
    results = json.load(f)

print(f"Model: {results['model']}")
print(f"Split: {results['split']}")
print(f"Subjects: {results['n_subjects']}")
print()

# Summary table
summary = results["summary"]
rows = []
for cls in ["NCR", "ED", "ET"]:
    s = summary[cls]
    rows.append({
        "Class": cls,
        "Dice mean": f"{s['dice_mean']:.4f}",
        "Dice std":  f"{s['dice_std']:.4f}",
        "HD95 mean": f"{s['hd95_mean']:.2f}" if s['hd95_mean'] is not None else "N/A",
        "HD95 std":  f"{s['hd95_std']:.2f}" if s['hd95_std'] is not None else "N/A",
        "HD95 NaN":  s['hd95_nan_count'],
    })

df_summary = pd.DataFrame(rows).set_index("Class")
print(df_summary.to_string())
print(f"\nMean fg Dice (3D): {summary['mean_fg_dice']:.4f}")

## 3 — Comparison with U-Net and SegFormer

In [ ]:
# Load all available results
models = {}
for name in ["unet", "fpn", "segformer"]:
    path = PROJECT_ROOT / "evaluation_outputs" / f"test_3d_metrics_{name}.json"
    if path.exists():
        with open(path) as f:
            models[name] = json.load(f)

if len(models) > 1:
    print("=" * 70)
    print("  3D TEST SET — MODEL COMPARISON")
    print("=" * 70)
    
    header = f"{'Model':<20s}"
    for cls in ["NCR", "ED", "ET"]:
        header += f"  {cls} Dice"
    header += "  Mean fg Dice"
    print(header)
    print("-" * len(header))
    
    display_names = {"unet": "U-Net/ResNet34", "fpn": "FPN/ResNet34", "segformer": "SegFormer-B1"}
    for name, data in models.items():
        s = data["summary"]
        row = f"{display_names.get(name, name):<20s}"
        for cls in ["NCR", "ED", "ET"]:
            row += f"  {s[cls]['dice_mean']:.4f}  "
        row += f"  {s['mean_fg_dice']:.4f}"
        print(row)
    print("=" * 70)
else:
    print("Only FPN results available. Run evaluation for other models to see comparison.")